In [1]:

# ============================================================
# THURSDAY - CELL 1: CHECK EXISTING FEATURE ENGINEERING
# Purpose:
# Inspect the current CA_1 dataset and identify which
# advanced time-series features are still missing.
# ============================================================

print("CURRENT FEATURE DATASET")
print("-----------------------")

print("Dataset shape:", ca1_features.shape)

print("\nExisting columns:")
for column in ca1_features.columns:
    print("-", column)

print("\nCURRENT FEATURE CHECK")
print("---------------------")

required_features = [
    "lag_28",
    "lag_35",
    "lag_42",
    "lag_56",
    "lag_364",
    "rolling_mean_7",
    "rolling_mean_28",
    "rolling_std_7",
    "rolling_std_28",
    "ema",
    "zero_sales_run",
    "day_of_week_sin",
    "day_of_week_cos",
    "month_sin",
    "month_cos",
    "price_momentum",
    "discount_ratio"
]

for feature in required_features:
    status = "AVAILABLE" if feature in ca1_features.columns else "MISSING"
    print(f"{feature}: {status}")

print("\nSTORE:", PILOT_STORE)
print("TOTAL ROWS:", ca1_features.height)


CURRENT FEATURE DATASET
-----------------------


NameError: name 'ca1_features' is not defined

In [2]:

# ============================================================
# THURSDAY - CELL 1: PROJECT SETUP
# Purpose:
# Import required libraries and define project folder paths.
# ============================================================

from pathlib import Path
import polars as pl
import pandas as pd
import numpy as np

# Define project directories.
PROJECT_DIR = Path(r"D:\Retail-Demand")

RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
MODEL_DIR = PROJECT_DIR / "models"
RESULTS_DIR = PROJECT_DIR / "results"
PREDICTION_DIR = PROJECT_DIR / "predictions"

# Select the pilot store.
PILOT_STORE = "CA_1"

# Define forecast horizon.
HORIZON = 28

print("PROJECT SETUP COMPLETED")
print("Project directory:", PROJECT_DIR)
print("Pilot store:", PILOT_STORE)
print("Forecast horizon:", HORIZON)


PROJECT SETUP COMPLETED
Project directory: D:\Retail-Demand
Pilot store: CA_1
Forecast horizon: 28


In [3]:

# ============================================================
# THURSDAY - CELL 2: CHECK SAVED DATASETS
# Purpose:
# Find previously saved Parquet datasets so we can reuse
# existing work instead of repeating data preparation.
# ============================================================

print("AVAILABLE PROCESSED DATASETS")
print("--------------------------------")

for file in PROCESSED_DATA_DIR.glob("*"):
    print(file.name, "-", round(file.stat().st_size / (1024**2), 2), "MB")

print("\nAVAILABLE PREDICTION FILES")
print("--------------------------------")

for file in PREDICTION_DIR.glob("*"):
    print(file.name)


AVAILABLE PROCESSED DATASETS
--------------------------------
m5_panel.parquet - 170.29 MB

AVAILABLE PREDICTION FILES
--------------------------------
conformal_predictions_ca1_fold3.parquet
croston_sba_fold_1.parquet
croston_sba_fold_2.parquet
croston_sba_fold_3.parquet
lightgbm_l2_ca1_fold1.parquet
lightgbm_tweedie_ca1_fold1.parquet
quantile_predictions_ca1_fold1.parquet
quantile_predictions_ca1_fold2.parquet
quantile_predictions_ca1_fold3.parquet
seasonal_naive_fold_1.parquet
seasonal_naive_fold_2.parquet
seasonal_naive_fold_3.parquet
temporal_calibrated_predictions_ca1_fold3.parquet


In [4]:

# ============================================================
# THURSDAY - CELL 3: LOAD CA_1 PANEL DATA
# Purpose:
# Load the saved M5 panel dataset and extract only CA_1.
# This avoids loading unnecessary stores into memory.
# ============================================================

PANEL_PATH = PROCESSED_DATA_DIR / "m5_panel.parquet"

# Read the Parquet file lazily and filter the pilot store.
ca1_panel = (
    pl.scan_parquet(PANEL_PATH)
    .filter(pl.col("store_id") == PILOT_STORE)
    .sort(["item_id", "date"])
    .collect()
)

print("CA_1 DATA LOADED SUCCESSFULLY")
print("--------------------------------")
print("Dataset shape:", ca1_panel.shape)

print("\nColumns:")
print(ca1_panel.columns)

print("\nDate range:")
print(ca1_panel["date"].min(), "to", ca1_panel["date"].max())

print("\nUnique products:", ca1_panel["item_id"].n_unique())
print("Unique stores:", ca1_panel["store_id"].n_unique())

print("\nSample records:")
display(ca1_panel.head(5))


CA_1 DATA LOADED SUCCESSFULLY
--------------------------------
Dataset shape: (5832737, 7)

Columns:
['item_id', 'store_id', 'date', 'sales', 'sell_price', 'event_name_1', 'snap_flag']

Date range:
2011-01-29 to 2016-04-24

Unique products: 3049
Unique stores: 1

Sample records:


item_id,store_id,date,sales,sell_price,event_name_1,snap_flag
str,str,date,i64,f64,str,i64
"""FOODS_1_001""","""CA_1""",2011-01-29,3,2.0,null,0
"""FOODS_1_001""","""CA_1""",2011-01-30,0,2.0,null,0
"""FOODS_1_001""","""CA_1""",2011-01-31,0,2.0,null,0
"""FOODS_1_001""","""CA_1""",2011-02-01,1,2.0,null,1
"""FOODS_1_001""","""CA_1""",2011-02-02,4,2.0,null,1


In [5]:

# ============================================================
# THURSDAY - CELL 4: ADVANCED LAG AND ROLLING FEATURES
# Purpose:
# Rebuild the earlier temporal features and add:
# 1. Lag 364
# 2. Rolling standard deviations
# 3. Cyclical day-of-week and month encodings
# All sales-based features respect the 28-day forecast gap.
# ============================================================

# Sort observations to ensure correct time-series calculations.
ca1_panel = ca1_panel.sort(["item_id", "date"])

# STEP 1: Create lag features and a 28-day shifted sales series.
ca1_features = ca1_panel.with_columns([

    pl.col("sales").shift(28).over("item_id").alias("lag_28"),
    pl.col("sales").shift(35).over("item_id").alias("lag_35"),
    pl.col("sales").shift(42).over("item_id").alias("lag_42"),
    pl.col("sales").shift(56).over("item_id").alias("lag_56"),
    pl.col("sales").shift(364).over("item_id").alias("lag_364"),

    pl.col("sales").shift(28).over("item_id").alias("_sales_gap28"),

    pl.col("date").dt.weekday().alias("day_of_week"),
    pl.col("date").dt.month().alias("month")
])

# STEP 2: Calculate rolling means and standard deviations
# using only sales observations available at least 28 days earlier.
ca1_features = ca1_features.with_columns([

    pl.col("_sales_gap28")
    .rolling_mean(window_size=7)
    .over("item_id")
    .alias("rolling_mean_7"),

    pl.col("_sales_gap28")
    .rolling_mean(window_size=28)
    .over("item_id")
    .alias("rolling_mean_28"),

    pl.col("_sales_gap28")
    .rolling_mean(window_size=56)
    .over("item_id")
    .alias("rolling_mean_56"),

    pl.col("_sales_gap28")
    .rolling_std(window_size=7)
    .over("item_id")
    .alias("rolling_std_7"),

    pl.col("_sales_gap28")
    .rolling_std(window_size=28)
    .over("item_id")
    .alias("rolling_std_28")
])

# STEP 3: Create cyclical calendar features.
# These represent repeating weekly and yearly patterns.

ca1_features = ca1_features.with_columns([

    (
        (2 * np.pi * (pl.col("day_of_week") - 1)) / 7
    ).sin().alias("day_of_week_sin"),

    (
        (2 * np.pi * (pl.col("day_of_week") - 1)) / 7
    ).cos().alias("day_of_week_cos"),

    (
        (2 * np.pi * (pl.col("month") - 1)) / 12
    ).sin().alias("month_sin"),

    (
        (2 * np.pi * (pl.col("month") - 1)) / 12
    ).cos().alias("month_cos")
])

# STEP 4: Remove the temporary column.
ca1_features = ca1_features.drop("_sales_gap28")

# STEP 5: Verify the resulting dataset.
print("ADVANCED FEATURE ENGINEERING COMPLETED")
print("---------------------------------------")

print("Dataset shape:", ca1_features.shape)

print("\nFeature columns:")
for column in ca1_features.columns:
    print("-", column)

print("\nNew feature null counts:")
new_features = [
    "lag_364",
    "rolling_std_7",
    "rolling_std_28",
    "day_of_week_sin",
    "day_of_week_cos",
    "month_sin",
    "month_cos"
]

print(
    ca1_features.select([
        pl.col(column).null_count().alias(column)
        for column in new_features
    ])
)


ADVANCED FEATURE ENGINEERING COMPLETED
---------------------------------------
Dataset shape: (5832737, 23)

Feature columns:
- item_id
- store_id
- date
- sales
- sell_price
- event_name_1
- snap_flag
- lag_28
- lag_35
- lag_42
- lag_56
- lag_364
- day_of_week
- month
- rolling_mean_7
- rolling_mean_28
- rolling_mean_56
- rolling_std_7
- rolling_std_28
- day_of_week_sin
- day_of_week_cos
- month_sin
- month_cos

New feature null counts:
shape: (1, 7)
┌─────────┬───────────────┬────────────────┬─────────────────┬─────────────────┬───────────┬───────────┐
│ lag_364 ┆ rolling_std_7 ┆ rolling_std_28 ┆ day_of_week_sin ┆ day_of_week_cos ┆ month_sin ┆ month_cos │
│ ---     ┆ ---           ┆ ---            ┆ ---             ┆ ---             ┆ ---       ┆ ---       │
│ u32     ┆ u32           ┆ u32            ┆ u32             ┆ u32             ┆ u32       ┆ u32       │
╞═════════╪═══════════════╪════════════════╪═════════════════╪═════════════════╪═══════════╪═══════════╡
│ 1109836 ┆ 103666 

In [6]:

# ============================================================
# THURSDAY - CELL 5: EMA AND ZERO-SALES RUN LENGTH
# Purpose:
# 1. Calculate an exponential moving average from sales
#    shifted by 28 days.
# 2. Count consecutive zero-sales days in the shifted history.
# 3. Preserve nulls where historical information is unavailable.
# ============================================================

# STEP 1: Recreate the sales series shifted by the forecast gap.
ca1_features = ca1_features.with_columns(
    pl.col("sales")
    .shift(28)
    .over("item_id")
    .alias("_sales_gap28")
)

# STEP 2: Calculate the exponential moving average.
# Span 28 gives more weight to recent observations.
ca1_features = ca1_features.with_columns(
    pl.col("_sales_gap28")
    .ewm_mean(span=28, adjust=False)
    .over("item_id")
    .alias("ema_28")
)

# STEP 3: Create a cumulative group number.
# Every positive sales observation starts a new zero-run group.
ca1_features = ca1_features.with_columns(
    (
        (pl.col("_sales_gap28") > 0)
        .fill_null(False)
        .cast(pl.UInt32)
        .cum_sum()
        .over("item_id")
    ).alias("_zero_group")
)

# STEP 4: Count consecutive zero-sales observations.
# A positive observation resets the run length to zero.
# Null shifted history remains null.
ca1_features = ca1_features.with_columns(
    pl.when(pl.col("_sales_gap28").is_null())
    .then(None)
    .when(pl.col("_sales_gap28") > 0)
    .then(0)
    .otherwise(
        pl.col("_sales_gap28")
        .cum_count()
        .over(["item_id", "_zero_group"])
        -
        pl.when(pl.col("_zero_group") > 0)
        .then(1)
        .otherwise(0)
    )
    .alias("zero_sales_run")
)

# STEP 5: Remove temporary calculation columns.
ca1_features = ca1_features.drop([
    "_sales_gap28",
    "_zero_group"
])

# STEP 6: Verify the new features.
print("EMA AND ZERO-SALES RUN FEATURES COMPLETED")
print("------------------------------------------")

print("Dataset shape:", ca1_features.shape)

print("\nNew feature null counts:")
print(
    ca1_features.select([
        pl.col("ema_28").null_count(),
        pl.col("zero_sales_run").null_count()
    ])
)

print("\nFeature summary:")
print(
    ca1_features.select([
        pl.col("ema_28").mean().alias("mean_ema"),
        pl.col("zero_sales_run").mean().alias("mean_zero_run"),
        pl.col("zero_sales_run").max().alias("maximum_zero_run")
    ])
)

print("\nSample records:")
display(
    ca1_features.select([
        "item_id",
        "date",
        "sales",
        "ema_28",
        "zero_sales_run"
    ]).filter(
        pl.col("item_id") == "FOODS_1_001"
    ).tail(10)
)


EMA AND ZERO-SALES RUN FEATURES COMPLETED
------------------------------------------
Dataset shape: (5832737, 25)

New feature null counts:
shape: (1, 2)
┌────────┬────────────────┐
│ ema_28 ┆ zero_sales_run │
│ ---    ┆ ---            │
│ u32    ┆ u32            │
╞════════╪════════════════╡
│ 85372  ┆ 85372          │
└────────┴────────────────┘

Feature summary:
shape: (1, 3)
┌──────────┬───────────────┬──────────────────┐
│ mean_ema ┆ mean_zero_run ┆ maximum_zero_run │
│ ---      ┆ ---           ┆ ---              │
│ f64      ┆ f64           ┆ u32              │
╞══════════╪═══════════════╪══════════════════╡
│ 1.316693 ┆ 105.330177    ┆ 1845             │
└──────────┴───────────────┴──────────────────┘

Sample records:


item_id,date,sales,ema_28,zero_sales_run
str,date,i64,f64,u32
"""FOODS_1_001""",2016-04-15,0,0.759157,0
"""FOODS_1_001""",2016-04-16,2,0.844732,0
"""FOODS_1_001""",2016-04-17,0,0.924406,0
"""FOODS_1_001""",2016-04-18,4,0.860654,1
"""FOODS_1_001""",2016-04-19,1,0.939229,0
"""FOODS_1_001""",2016-04-20,1,1.012386,0
"""FOODS_1_001""",2016-04-21,0,0.942566,1
"""FOODS_1_001""",2016-04-22,1,0.946527,0
"""FOODS_1_001""",2016-04-23,1,0.950215,0


In [7]:

# ============================================================
# THURSDAY - CELL 6: PRICE MOMENTUM AND DISCOUNT RATIO
# Purpose:
# 1. Calculate historical price momentum.
# 2. Calculate the discount ratio against a historical price peak.
# 3. Avoid using prices from inside the 28-day forecast gap.
# ============================================================

# STEP 1: Create historical price features.
ca1_features = ca1_features.with_columns([

    # Price available 28 days before the forecast date.
    pl.col("sell_price")
    .shift(28)
    .over("item_id")
    .alias("price_lag_28"),

    # Price available 56 days before the forecast date.
    pl.col("sell_price")
    .shift(56)
    .over("item_id")
    .alias("price_lag_56")
])

# STEP 2: Calculate price momentum.
# Positive value = historical price increased.
# Negative value = historical price decreased.

ca1_features = ca1_features.with_columns(

    pl.when(
        pl.col("price_lag_56").is_not_null() &
        (pl.col("price_lag_56") > 0)
    )
    .then(
        (pl.col("price_lag_28") / pl.col("price_lag_56")) - 1
    )
    .otherwise(None)
    .alias("price_momentum")
)

# STEP 3: Calculate the historical maximum price.
# Use the price series shifted by 28 days.
ca1_features = ca1_features.with_columns(

    pl.col("sell_price")
    .shift(28)
    .rolling_max(window_size=56)
    .over("item_id")
    .alias("historical_max_price_56")
)

# STEP 4: Calculate discount ratio.
# A value below 1 indicates the historical price was
# below its historical maximum.

ca1_features = ca1_features.with_columns(

    pl.when(
        pl.col("historical_max_price_56").is_not_null() &
        (pl.col("historical_max_price_56") > 0)
    )
    .then(
        pl.col("price_lag_28") /
        pl.col("historical_max_price_56")
    )
    .otherwise(None)
    .alias("discount_ratio")
)

# STEP 5: Verify the new features.
print("PRICE FEATURE ENGINEERING COMPLETED")
print("------------------------------------")

print("Dataset shape:", ca1_features.shape)

print("\nPrice feature null counts:")

print(
    ca1_features.select([
        pl.col("price_lag_28").null_count(),
        pl.col("price_lag_56").null_count(),
        pl.col("price_momentum").null_count(),
        pl.col("historical_max_price_56").null_count(),
        pl.col("discount_ratio").null_count()
    ])
)

print("\nPrice feature summary:")

print(
    ca1_features.select([
        pl.col("price_momentum").mean().alias("mean_price_momentum"),
        pl.col("discount_ratio").mean().alias("mean_discount_ratio"),
        pl.col("discount_ratio").min().alias("minimum_discount_ratio"),
        pl.col("discount_ratio").max().alias("maximum_discount_ratio")
    ])
)

print("\nSample price records:")

display(
    ca1_features.select([
        "item_id",
        "date",
        "sell_price",
        "price_lag_28",
        "price_lag_56",
        "price_momentum",
        "discount_ratio"
    ])
    .filter(pl.col("item_id") == "FOODS_1_001")
    .tail(10)
)


PRICE FEATURE ENGINEERING COMPLETED
------------------------------------
Dataset shape: (5832737, 30)

Price feature null counts:
shape: (1, 5)
┌──────────────┬──────────────┬────────────────┬─────────────────────────┬────────────────┐
│ price_lag_28 ┆ price_lag_56 ┆ price_momentum ┆ historical_max_price_56 ┆ discount_ratio │
│ ---          ┆ ---          ┆ ---            ┆ ---                     ┆ ---            │
│ u32          ┆ u32          ┆ u32            ┆ u32                     ┆ u32            │
╞══════════════╪══════════════╪════════════════╪═════════════════════════╪════════════════╡
│ 1215214      ┆ 1300586      ┆ 1300586        ┆ 1382898                 ┆ 1382898        │
└──────────────┴──────────────┴────────────────┴─────────────────────────┴────────────────┘

Price feature summary:
shape: (1, 4)
┌─────────────────────┬─────────────────────┬────────────────────────┬────────────────────────┐
│ mean_price_momentum ┆ mean_discount_ratio ┆ minimum_discount_ratio ┆ maximum

item_id,date,sell_price,price_lag_28,price_lag_56,price_momentum,discount_ratio
str,date,f64,f64,f64,f64,f64
"""FOODS_1_001""",2016-04-15,2.24,2.24,2.24,0.0,1.0
"""FOODS_1_001""",2016-04-16,2.24,2.24,2.24,0.0,1.0
"""FOODS_1_001""",2016-04-17,2.24,2.24,2.24,0.0,1.0
"""FOODS_1_001""",2016-04-18,2.24,2.24,2.24,0.0,1.0
"""FOODS_1_001""",2016-04-19,2.24,2.24,2.24,0.0,1.0
"""FOODS_1_001""",2016-04-20,2.24,2.24,2.24,0.0,1.0
"""FOODS_1_001""",2016-04-21,2.24,2.24,2.24,0.0,1.0
"""FOODS_1_001""",2016-04-22,2.24,2.24,2.24,0.0,1.0
"""FOODS_1_001""",2016-04-23,2.24,2.24,2.24,0.0,1.0


In [8]:

# ============================================================
# THURSDAY - CELL 7: FEATURE QUALITY VALIDATION
# Purpose:
# Check feature nulls, invalid values, duplicate observations,
# and price feature ranges before saving the dataset.
# ============================================================

print("FEATURE DATASET VALIDATION")
print("===========================")

print("\n1. Dataset information")
print("Rows:", ca1_features.height)
print("Columns:", len(ca1_features.columns))
print("Unique products:", ca1_features["item_id"].n_unique())

# Check duplicate product-date combinations.
duplicate_count = (
    ca1_features
    .group_by(["item_id", "date"])
    .len()
    .filter(pl.col("len") > 1)
    .height
)

print("\n2. Duplicate product-date records:", duplicate_count)

# Check invalid numerical values.
print("\n3. Invalid value checks")

print(
    "Negative sales:",
    ca1_features.filter(pl.col("sales") < 0).height
)

print(
    "Negative EMA:",
    ca1_features.filter(pl.col("ema_28") < 0).height
)

print(
    "Negative rolling standard deviation:",
    ca1_features.filter(
        (pl.col("rolling_std_7") < 0) |
        (pl.col("rolling_std_28") < 0)
    ).height
)

print(
    "Negative price momentum:",
    ca1_features.filter(
        pl.col("price_momentum") < -1
    ).height
)

print(
    "Discount ratio above 1:",
    ca1_features.filter(
        pl.col("discount_ratio") > 1
    ).height
)

print("\n4. Feature null counts")

important_features = [
    "lag_28", "lag_364",
    "rolling_mean_7", "rolling_mean_28",
    "rolling_std_7", "rolling_std_28",
    "ema_28", "zero_sales_run",
    "price_momentum", "discount_ratio"
]

display(
    ca1_features.select([
        pl.col(column).null_count().alias(column)
        for column in important_features
    ])
)

print("\n5. Final feature dataset shape:", ca1_features.shape)


FEATURE DATASET VALIDATION

1. Dataset information
Rows: 5832737
Columns: 30
Unique products: 3049

2. Duplicate product-date records: 0

3. Invalid value checks
Negative sales: 0
Negative EMA: 0
Negative rolling standard deviation: 0
Negative price momentum: 0
Discount ratio above 1: 0

4. Feature null counts


lag_28,lag_364,rolling_mean_7,rolling_mean_28,rolling_std_7,rolling_std_28,ema_28,zero_sales_run,price_momentum,discount_ratio
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
85372,1109836,103666,167695,103666,167695,85372,85372,1300586,1382898



5. Final feature dataset shape: (5832737, 30)


In [9]:

# ============================================================
# THURSDAY - CELL 8: SAVE ADVANCED FEATURE DATASET
# Purpose:
# Save the completed CA_1 feature dataset in Parquet format
# and verify that the saved file can be read successfully.
# ============================================================

FEATURE_PATH = PROCESSED_DATA_DIR / "ca1_advanced_features.parquet"

# Save the feature dataset with compression.
ca1_features.write_parquet(
    FEATURE_PATH,
    compression="zstd"
)

print("FEATURE DATASET SAVED")
print("---------------------")
print("File:", FEATURE_PATH)
print("Size (MB):", round(FEATURE_PATH.stat().st_size / (1024**2), 2))

# Reload lazily and verify the saved dataset.
saved_features = pl.scan_parquet(FEATURE_PATH)

print("\nSAVED DATASET VERIFICATION")
print("--------------------------")
print("Rows:", saved_features.select(pl.len()).collect().item())
print("Columns:", len(saved_features.collect_schema().names()))
print("Columns:", saved_features.collect_schema().names())

print("\nFEATURE DATASET READY FOR MODEL TRAINING!")


FEATURE DATASET SAVED
---------------------
File: D:\Retail-Demand\data\processed\ca1_advanced_features.parquet
Size (MB): 72.41

SAVED DATASET VERIFICATION
--------------------------
Rows: 5832737
Columns: 30
Columns: ['item_id', 'store_id', 'date', 'sales', 'sell_price', 'event_name_1', 'snap_flag', 'lag_28', 'lag_35', 'lag_42', 'lag_56', 'lag_364', 'day_of_week', 'month', 'rolling_mean_7', 'rolling_mean_28', 'rolling_mean_56', 'rolling_std_7', 'rolling_std_28', 'day_of_week_sin', 'day_of_week_cos', 'month_sin', 'month_cos', 'ema_28', 'zero_sales_run', 'price_lag_28', 'price_lag_56', 'price_momentum', 'historical_max_price_56', 'discount_ratio']

FEATURE DATASET READY FOR MODEL TRAINING!


In [10]:

# ============================================================
# THURSDAY - CELL 9: DEFINE ROLLING-ORIGIN FOLDS
# Purpose:
# Define the three 28-day validation periods and create
# chronological training and validation boundaries.
# ============================================================

fold_definitions = {
    1: {
        "train_end": pd.Timestamp("2016-01-31"),
        "validation_start": pd.Timestamp("2016-02-01"),
        "validation_end": pd.Timestamp("2016-02-28")
    },
    2: {
        "train_end": pd.Timestamp("2016-02-28"),
        "validation_start": pd.Timestamp("2016-02-29"),
        "validation_end": pd.Timestamp("2016-03-27")
    },
    3: {
        "train_end": pd.Timestamp("2016-03-27"),
        "validation_start": pd.Timestamp("2016-03-28"),
        "validation_end": pd.Timestamp("2016-04-24")
    }
}

# The final 28 days of each training window are used
# as internal validation for early stopping.
INNER_VALIDATION_DAYS = 28

print("ROLLING-ORIGIN FOLD DEFINITIONS")
print("--------------------------------")

for fold_number, dates in fold_definitions.items():

    print(f"\nFOLD {fold_number}")
    print("Training ends:", dates["train_end"].date())
    print("Outer validation:",
          dates["validation_start"].date(),
          "to",
          dates["validation_end"].date())

print("\nInternal early-stopping window:",
      INNER_VALIDATION_DAYS, "days")

ROLLING-ORIGIN FOLD DEFINITIONS
--------------------------------

FOLD 1
Training ends: 2016-01-31
Outer validation: 2016-02-01 to 2016-02-28

FOLD 2
Training ends: 2016-02-28
Outer validation: 2016-02-29 to 2016-03-27

FOLD 3
Training ends: 2016-03-27
Outer validation: 2016-03-28 to 2016-04-24

Internal early-stopping window: 28 days


In [11]:

# ============================================================
# THURSDAY - CELL 10: SELECT FEATURES AND CHECK FOLD 1
# Purpose:
# 1. Define the features used by the advanced models.
# 2. Separate categorical and numerical features.
# 3. Define the internal early-stopping period.
# 4. Verify training and validation row counts.
# ============================================================

from datetime import timedelta

# STEP 1: Define model features.
# Current sell_price is intentionally excluded.
# Historical price features are used instead.

feature_columns = [
    "item_id",
    "event_name_1",
    "day_of_week",
    "month",
    "snap_flag",

    "lag_28",
    "lag_35",
    "lag_42",
    "lag_56",
    "lag_364",

    "rolling_mean_7",
    "rolling_mean_28",
    "rolling_mean_56",

    "rolling_std_7",
    "rolling_std_28",

    "day_of_week_sin",
    "day_of_week_cos",
    "month_sin",
    "month_cos",

    "ema_28",
    "zero_sales_run",

    "price_lag_28",
    "price_lag_56",
    "price_momentum",
    "historical_max_price_56",
    "discount_ratio"
]

categorical_features = [
    "item_id",
    "event_name_1",
    "day_of_week",
    "month"
]

numeric_features = [
    column for column in feature_columns
    if column not in categorical_features
]

print("MODEL FEATURE CONFIGURATION")
print("--------------------------------")

print("Total features:", len(feature_columns))
print("Categorical features:", len(categorical_features))
print("Numerical features:", len(numeric_features))

# STEP 2: Define Fold 1 boundaries.
fold_number = 1
fold_dates = fold_definitions[fold_number]

outer_train_end = fold_dates["train_end"].date()
outer_validation_start = fold_dates["validation_start"].date()
outer_validation_end = fold_dates["validation_end"].date()

# Reserve the final 28 days of the outer training period
# for internal early stopping.
inner_validation_start = (
    outer_train_end - timedelta(days=INNER_VALIDATION_DAYS - 1)
)

inner_train_end = inner_validation_start - timedelta(days=1)

print("\nFOLD 1 DATE BOUNDARIES")
print("--------------------------------")

print("Internal training ends:", inner_train_end)
print("Internal validation:",
      inner_validation_start, "to", outer_train_end)

print("Outer validation:",
      outer_validation_start, "to", outer_validation_end)

# STEP 3: Count rows for each partition.
def count_rows(start_date, end_date):
    return (
        ca1_features
        .filter(
            (pl.col("date") >= start_date) &
            (pl.col("date") <= end_date)
        )
        .height
    )

inner_train_rows = count_rows(
    ca1_features["date"].min(),
    inner_train_end
)

inner_validation_rows = count_rows(
    inner_validation_start,
    outer_train_end
)

outer_validation_rows = count_rows(
    outer_validation_start,
    outer_validation_end
)

print("\nFOLD 1 ROW COUNTS")
print("--------------------------------")
print("Internal training rows:", inner_train_rows)
print("Internal validation rows:", inner_validation_rows)
print("Outer validation rows:", outer_validation_rows)

# STEP 4: Verify required features exist.
missing_columns = [
    column for column in feature_columns
    if column not in ca1_features.columns
]

print("\nMISSING MODEL FEATURES:", missing_columns)

assert len(missing_columns) == 0
assert inner_validation_rows == 3049 * 28
assert outer_validation_rows == 3049 * 28

print("\nFOLD 1 FEATURE CONFIGURATION VERIFIED!")


MODEL FEATURE CONFIGURATION
--------------------------------
Total features: 26
Categorical features: 4
Numerical features: 22

FOLD 1 DATE BOUNDARIES
--------------------------------
Internal training ends: 2016-01-03
Internal validation: 2016-01-04 to 2016-01-31
Outer validation: 2016-02-01 to 2016-02-28

FOLD 1 ROW COUNTS
--------------------------------
Internal training rows: 5491249
Internal validation rows: 85372
Outer validation rows: 85372

MISSING MODEL FEATURES: []

FOLD 1 FEATURE CONFIGURATION VERIFIED!


In [12]:

# ============================================================
# THURSDAY - CELL 11: PREPARE FOLD 1 MODEL DATASETS
# Purpose:
# 1. Extract the three chronological data partitions.
# 2. Separate features (X) and actual sales (y).
# 3. Apply consistent categorical encodings.
# 4. Verify data types, shapes and missing values.
# ============================================================

# STEP 1: Define a helper to extract a date range.
def prepare_date_range(start_date, end_date):

    return (
        ca1_features
        .filter(
            (pl.col("date") >= start_date) &
            (pl.col("date") <= end_date)
        )
        .select(["date", "sales"] + feature_columns)
        .to_pandas()
    )

# STEP 2: Load the three chronological partitions.
train_data = prepare_date_range(
    ca1_features["date"].min(),
    inner_train_end
)

early_stop_data = prepare_date_range(
    inner_validation_start,
    outer_train_end
)

outer_test_data = prepare_date_range(
    outer_validation_start,
    outer_validation_end
)

print("DATA PARTITIONS LOADED")
print("--------------------------------")
print("Training:", train_data.shape)
print("Early stopping:", early_stop_data.shape)
print("Outer validation:", outer_test_data.shape)

# STEP 3: Separate features and target.
X_train = train_data[feature_columns].copy()
y_train = train_data["sales"].to_numpy(dtype=np.float32)

X_early_stop = early_stop_data[feature_columns].copy()
y_early_stop = early_stop_data["sales"].to_numpy(dtype=np.float32)

X_outer = outer_test_data[feature_columns].copy()
y_outer = outer_test_data["sales"].to_numpy(dtype=np.float32)

# STEP 4: Establish category levels from training data.
# Apply exactly the same category levels to both validation sets.
category_types = {}

for column in categorical_features:

    categories = pd.Index(
        X_train[column].dropna().unique()
    )

    category_types[column] = pd.CategoricalDtype(
        categories=categories
    )

    X_train[column] = X_train[column].astype(category_types[column])
    X_early_stop[column] = X_early_stop[column].astype(category_types[column])
    X_outer[column] = X_outer[column].astype(category_types[column])

# STEP 5: Convert numerical features to float32.
for column in numeric_features:

    X_train[column] = X_train[column].astype("float32")
    X_early_stop[column] = X_early_stop[column].astype("float32")
    X_outer[column] = X_outer[column].astype("float32")

# STEP 6: Verify feature types and missing values.
print("\nFEATURE DATA TYPES")
print("--------------------------------")
print(X_train.dtypes)

print("\nMISSING VALUES")
print("--------------------------------")
print("Training:", X_train.isna().sum().sum())
print("Early stopping:", X_early_stop.isna().sum().sum())
print("Outer validation:", X_outer.isna().sum().sum())

print("\nFINAL DATASET SHAPES")
print("--------------------------------")
print("X_train:", X_train.shape)
print("X_early_stop:", X_early_stop.shape)
print("X_outer:", X_outer.shape)

print("\nTARGET SHAPES")
print("y_train:", y_train.shape)
print("y_early_stop:", y_early_stop.shape)
print("y_outer:", y_outer.shape)

assert list(X_train.columns) == feature_columns
assert list(X_early_stop.columns) == feature_columns
assert list(X_outer.columns) == feature_columns

assert len(X_train) == len(y_train)
assert len(X_early_stop) == len(y_early_stop)
assert len(X_outer) == len(y_outer)

print("\nFOLD 1 MODEL DATA PREPARATION COMPLETED!")


DATA PARTITIONS LOADED
--------------------------------
Training: (5491249, 28)
Early stopping: (85372, 28)
Outer validation: (85372, 28)

FEATURE DATA TYPES
--------------------------------
item_id                    category
event_name_1               category
day_of_week                category
month                      category
snap_flag                   float32
lag_28                      float32
lag_35                      float32
lag_42                      float32
lag_56                      float32
lag_364                     float32
rolling_mean_7              float32
rolling_mean_28             float32
rolling_mean_56             float32
rolling_std_7               float32
rolling_std_28              float32
day_of_week_sin             float32
day_of_week_cos             float32
month_sin                   float32
month_cos                   float32
ema_28                      float32
zero_sales_run              float32
price_lag_28                float32
price_lag_56     

In [13]:

# ============================================================
# THURSDAY - CELL 12: TRAIN ADVANCED LIGHTGBM L2 MODEL
# Purpose:
# 1. Train LightGBM using squared-error regression.
# 2. Use internal validation for early stopping.
# 3. Evaluate predictions on the untouched outer validation set.
# 4. Save the trained model and prediction results.
# ============================================================

import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error

# STEP 1: Configure the LightGBM L2 model.
l2_model = lgb.LGBMRegressor(
    objective="regression_l2",
    n_estimators=700,
    learning_rate=0.05,
    num_leaves=31,
    max_bin=127,
    min_child_samples=100,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    n_jobs=4,
    random_state=42,
    verbosity=-1
)

print("LIGHTGBM L2 TRAINING STARTED")
print("--------------------------------")
print("Training rows:", len(X_train))
print("Internal validation rows:", len(X_early_stop))
print("Features:", len(feature_columns))

# STEP 2: Train with internal validation and early stopping.
l2_model.fit(
    X_train,
    y_train,
    eval_set=[(X_early_stop, y_early_stop)],
    eval_metric="rmse",
    categorical_feature=categorical_features,
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=50)
    ]
)

print("\nTRAINING COMPLETED")
print("Best iteration:", l2_model.best_iteration_)

# STEP 3: Predict on the untouched outer validation set.
l2_predictions = l2_model.predict(
    X_outer,
    num_iteration=l2_model.best_iteration_
)

l2_predictions = np.maximum(l2_predictions, 0)

# STEP 4: Calculate evaluation metrics.
l2_mae = mean_absolute_error(y_outer, l2_predictions)

l2_rmse = np.sqrt(
    mean_squared_error(y_outer, l2_predictions)
)

l2_wape = (
    np.sum(np.abs(y_outer - l2_predictions)) /
    np.sum(np.abs(y_outer))
) * 100

l2_bias = (
    np.sum(l2_predictions - y_outer) /
    np.sum(y_outer)
) * 100

print("\nLIGHTGBM L2 OUTER VALIDATION RESULTS")
print("--------------------------------")
print("MAE:", round(l2_mae, 4))
print("RMSE:", round(l2_rmse, 4))
print("WAPE (%):", round(l2_wape, 4))
print("Bias (%):", round(l2_bias, 4))

# STEP 5: Save the trained model.
l2_model_path = MODEL_DIR / "lightgbm_l2_advanced_ca1_fold1.txt"

l2_model.booster_.save_model(str(l2_model_path))

print("\nMODEL SAVED:", l2_model_path)

# STEP 6: Save outer validation predictions.
l2_output = outer_test_data[["date", "sales", "item_id"]].copy()

l2_output = l2_output.rename(columns={
    "date": "forecast_date",
    "sales": "actual"
})

l2_output["prediction"] = l2_predictions
l2_output["horizon"] = (
    (pd.to_datetime(l2_output["forecast_date"]) -
     pd.Timestamp(outer_validation_start)).dt.days + 1
)

l2_prediction_path = (
    PREDICTION_DIR / "lightgbm_l2_advanced_ca1_fold1.parquet"
)

l2_output.to_parquet(l2_prediction_path, index=False)

print("PREDICTIONS SAVED:", l2_prediction_path)
print("Prediction rows:", len(l2_output))

print("\nADVANCED LIGHTGBM L2 COMPLETED!")


LIGHTGBM L2 TRAINING STARTED
--------------------------------
Training rows: 5491249
Internal validation rows: 85372
Features: 26


D:\Retail-Demand\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 50 rounds
[50]	valid_0's rmse: 2.15549	valid_0's l2: 4.64615
[100]	valid_0's rmse: 2.14085	valid_0's l2: 4.58324
[150]	valid_0's rmse: 2.14271	valid_0's l2: 4.5912
Early stopping, best iteration is:
[100]	valid_0's rmse: 2.14085	valid_0's l2: 4.58324

TRAINING COMPLETED
Best iteration: 100

LIGHTGBM L2 OUTER VALIDATION RESULTS
--------------------------------
MAE: 1.11
RMSE: 2.2052
WAPE (%): 77.1289
Bias (%): -2.6981

MODEL SAVED: D:\Retail-Demand\models\lightgbm_l2_advanced_ca1_fold1.txt
PREDICTIONS SAVED: D:\Retail-Demand\predictions\lightgbm_l2_advanced_ca1_fold1.parquet
Prediction rows: 85372

ADVANCED LIGHTGBM L2 COMPLETED!


In [14]:
# CELL 13: Train the advanced LightGBM Tweedie model for CA_1, Fold 1

# Import LightGBM and evaluation metrics
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Create the Tweedie regression model
tweedie_model = lgb.LGBMRegressor(
    objective="tweedie",
    tweedie_variance_power=1.2,
    n_estimators=700,
    learning_rate=0.05,
    num_leaves=31,
    max_bin=127,
    min_child_samples=100,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    n_jobs=4,
    random_state=42,
    verbosity=-1
)

# Train using the internal validation period for early stopping
print("LIGHTGBM TWEEDIE TRAINING STARTED")
print("--------------------------------")
print("Training rows:", len(X_train))
print("Internal validation rows:", len(X_early_stop))
print("Features:", len(feature_columns))

tweedie_model.fit(
    X_train,
    y_train,
    eval_set=[(X_early_stop, y_early_stop)],
    eval_metric="rmse",
    categorical_feature=categorical_features,
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=50)
    ]
)

# Predict on the untouched outer validation period
tweedie_predictions = tweedie_model.predict(
    X_outer,
    num_iteration=tweedie_model.best_iteration_
)

# Demand cannot be negative, so clip negative predictions to zero
tweedie_predictions = np.maximum(tweedie_predictions, 0)

# Calculate outer validation metrics
tweedie_mae = mean_absolute_error(y_outer, tweedie_predictions)
tweedie_rmse = np.sqrt(mean_squared_error(y_outer, tweedie_predictions))
tweedie_wape = (
    np.sum(np.abs(y_outer - tweedie_predictions))
    / np.sum(np.abs(y_outer))
) * 100
tweedie_bias = (
    np.sum(tweedie_predictions - y_outer)
    / np.sum(y_outer)
) * 100

# Display model performance
print("\nLIGHTGBM TWEEDIE OUTER VALIDATION RESULTS")
print("--------------------------------")
print("Best iteration:", tweedie_model.best_iteration_)
print(f"MAE: {tweedie_mae:.4f}")
print(f"RMSE: {tweedie_rmse:.4f}")
print(f"WAPE (%): {tweedie_wape:.4f}")
print(f"Bias (%): {tweedie_bias:.4f}")

# Save the trained Tweedie model
tweedie_model_path = MODEL_DIR / "lightgbm_tweedie_advanced_ca1_fold1.txt"
tweedie_model.booster_.save_model(str(tweedie_model_path))

# Save predictions with actual values and forecast horizon
tweedie_output = outer_test_data[
    ["date", "sales", "item_id"]
].copy()

tweedie_output = tweedie_output.rename(
    columns={
        "date": "forecast_date",
        "sales": "actual"
    }
)

tweedie_output["prediction"] = tweedie_predictions
tweedie_output["horizon"] = (
    pd.to_datetime(tweedie_output["forecast_date"])
    - fold_definitions[1]["validation_start"]
).dt.days + 1

tweedie_prediction_path = (
    PREDICTION_DIR / "lightgbm_tweedie_advanced_ca1_fold1.parquet"
)

tweedie_output.to_parquet(tweedie_prediction_path, index=False)

print("\nMODEL SAVED:", tweedie_model_path)
print("PREDICTIONS SAVED:", tweedie_prediction_path)
print("Prediction rows:", len(tweedie_output))
print("\nADVANCED LIGHTGBM TWEEDIE COMPLETED!")

LIGHTGBM TWEEDIE TRAINING STARTED
--------------------------------
Training rows: 5491249
Internal validation rows: 85372
Features: 26


D:\Retail-Demand\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 50 rounds
[50]	valid_0's rmse: 2.17411	valid_0's tweedie: 7.22942
[100]	valid_0's rmse: 2.14178	valid_0's tweedie: 7.20929
[150]	valid_0's rmse: 2.14163	valid_0's tweedie: 7.20504
Early stopping, best iteration is:
[122]	valid_0's rmse: 2.14086	valid_0's tweedie: 7.20671

LIGHTGBM TWEEDIE OUTER VALIDATION RESULTS
--------------------------------
Best iteration: 122
MAE: 1.0999
RMSE: 2.1974
WAPE (%): 76.4282
Bias (%): -3.1261

MODEL SAVED: D:\Retail-Demand\models\lightgbm_tweedie_advanced_ca1_fold1.txt
PREDICTIONS SAVED: D:\Retail-Demand\predictions\lightgbm_tweedie_advanced_ca1_fold1.parquet
Prediction rows: 85372

ADVANCED LIGHTGBM TWEEDIE COMPLETED!


In [15]:
# CELL 14: Train LightGBM quantile models for lower and upper demand estimates

# Import LightGBM
import lightgbm as lgb
import numpy as np
import pandas as pd

# Store the two quantile models
quantile_models = {}
quantile_predictions = {}

# Train one model for each quantile
for alpha in [0.05, 0.95]:

    print(f"\nTRAINING QUANTILE MODEL: {alpha}")
    print("--------------------------------")

    # Configure the quantile regression model
    model = lgb.LGBMRegressor(
        objective="quantile",
        alpha=alpha,
        n_estimators=700,
        learning_rate=0.05,
        num_leaves=31,
        max_bin=127,
        min_child_samples=100,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        n_jobs=4,
        random_state=42,
        verbosity=-1
    )

    # Train using the internal validation period
    model.fit(
        X_train,
        y_train,
        eval_set=[(X_early_stop, y_early_stop)],
        eval_metric="quantile",
        categorical_feature=categorical_features,
        callbacks=[
            lgb.early_stopping(stopping_rounds=50),
            lgb.log_evaluation(period=50)
        ]
    )

    # Generate predictions for the outer validation period
    predictions = model.predict(
        X_outer,
        num_iteration=model.best_iteration_
    )

    # Store the model and predictions
    quantile_models[alpha] = model
    quantile_predictions[alpha] = predictions

    print(f"Best iteration: {model.best_iteration_}")
    print(f"Predictions generated: {len(predictions)}")

    # Save the trained model
    model_path = MODEL_DIR / f"lightgbm_quantile_{alpha}_advanced_ca1_fold1.txt"
    model.booster_.save_model(str(model_path))

    print("Model saved:", model_path)

# Organize lower and upper predictions
lower_predictions = np.maximum(quantile_predictions[0.05], 0)
upper_predictions = np.maximum(quantile_predictions[0.95], 0)

# Ensure lower bounds do not exceed upper bounds
lower_bound = np.minimum(lower_predictions, upper_predictions)
upper_bound = np.maximum(lower_predictions, upper_predictions)

# Calculate prediction interval metrics
actual_values = np.asarray(y_outer)

coverage = np.mean(
    (actual_values >= lower_bound) &
    (actual_values <= upper_bound)
) * 100

average_width = np.mean(upper_bound - lower_bound)

lower_crossing_count = np.sum(lower_predictions > upper_predictions)

print("\nQUANTILE MODEL RESULTS")
print("--------------------------------")
print(f"Interval coverage: {coverage:.4f}%")
print(f"Average interval width: {average_width:.4f}")
print(f"Crossed intervals before correction: {lower_crossing_count}")
print("Expected nominal coverage: 90%")

# Save the prediction intervals
quantile_output = outer_test_data[
    ["date", "sales", "item_id"]
].copy()

quantile_output = quantile_output.rename(
    columns={
        "date": "forecast_date",
        "sales": "actual"
    }
)

quantile_output["lower_bound"] = lower_bound
quantile_output["upper_bound"] = upper_bound

quantile_output["horizon"] = (
    pd.to_datetime(quantile_output["forecast_date"])
    - fold_definitions[1]["validation_start"]
).dt.days + 1

quantile_output_path = (
    PREDICTION_DIR / "quantile_advanced_ca1_fold1.parquet"
)

quantile_output.to_parquet(quantile_output_path, index=False)

print("\nPREDICTIONS SAVED:", quantile_output_path)
print("Prediction rows:", len(quantile_output))
print("\nQUANTILE REGRESSION COMPLETED!")


TRAINING QUANTILE MODEL: 0.05
--------------------------------


D:\Retail-Demand\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 50 rounds
[50]	valid_0's quantile: 0.0686705
[100]	valid_0's quantile: 0.0686179
[150]	valid_0's quantile: 0.0685658
[200]	valid_0's quantile: 0.0685587
[250]	valid_0's quantile: 0.068555
[300]	valid_0's quantile: 0.068554
[350]	valid_0's quantile: 0.0685575
Early stopping, best iteration is:
[304]	valid_0's quantile: 0.068553
Best iteration: 304
Predictions generated: 85372
Model saved: D:\Retail-Demand\models\lightgbm_quantile_0.05_advanced_ca1_fold1.txt

TRAINING QUANTILE MODEL: 0.95
--------------------------------


D:\Retail-Demand\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 50 rounds
[50]	valid_0's quantile: 0.222033
[100]	valid_0's quantile: 0.216268
[150]	valid_0's quantile: 0.214914
[200]	valid_0's quantile: 0.214308
[250]	valid_0's quantile: 0.213872
Early stopping, best iteration is:
[242]	valid_0's quantile: 0.213823
Best iteration: 242
Predictions generated: 85372
Model saved: D:\Retail-Demand\models\lightgbm_quantile_0.95_advanced_ca1_fold1.txt

QUANTILE MODEL RESULTS
--------------------------------
Interval coverage: 94.4619%
Average interval width: 4.2474
Crossed intervals before correction: 0
Expected nominal coverage: 90%

PREDICTIONS SAVED: D:\Retail-Demand\predictions\quantile_advanced_ca1_fold1.parquet
Prediction rows: 85372

QUANTILE REGRESSION COMPLETED!


In [16]:
# CELL 15: Generate advanced quantile predictions for Fold 2 and Fold 3

# Function to prepare the feature data for a selected validation fold
def prepare_quantile_fold(fold_number):

    # Get the validation date range for this fold
    fold_info = fold_definitions[fold_number]
    start_date = fold_info["validation_start"]
    end_date = fold_info["validation_end"]

    # Select only the rows belonging to this validation period
    fold_data = (
        ca1_features
        .filter(
            (pl.col("date") >= start_date) &
            (pl.col("date") <= end_date)
        )
        .select(["date", "sales"] + feature_columns)
        .to_pandas()
    )

    # Separate model features from actual sales
    X_fold = fold_data[feature_columns].copy()

    # Apply the same categorical levels used during model training
    for column in categorical_features:
        X_fold[column] = pd.Categorical(
            X_fold[column],
            categories=X_train[column].cat.categories
        )

    # Convert numerical features to the same data type used during training
    for column in numeric_features:
        X_fold[column] = pd.to_numeric(
            X_fold[column],
            errors="coerce"
        ).astype(np.float32)

    return fold_data, X_fold


# Generate predictions for Fold 2 and Fold 3
for fold_number in [2, 3]:

    print(f"\nGENERATING QUANTILE PREDICTIONS: FOLD {fold_number}")
    print("--------------------------------")

    # Prepare the validation data
    fold_data, X_fold = prepare_quantile_fold(fold_number)

    # Generate lower and upper quantile predictions
    lower_raw = quantile_models[0.05].predict(
        X_fold,
        num_iteration=quantile_models[0.05].best_iteration_
    )

    upper_raw = quantile_models[0.95].predict(
        X_fold,
        num_iteration=quantile_models[0.95].best_iteration_
    )

    # Clip negative demand estimates and correct any crossed intervals
    lower_predictions_fold = np.maximum(lower_raw, 0)
    upper_predictions_fold = np.maximum(upper_raw, 0)

    lower_bound_fold = np.minimum(
        lower_predictions_fold,
        upper_predictions_fold
    )

    upper_bound_fold = np.maximum(
        lower_predictions_fold,
        upper_predictions_fold
    )

    # Create the prediction output
    fold_output = fold_data[["date", "sales", "item_id"]].copy()

    fold_output = fold_output.rename(
        columns={
            "date": "forecast_date",
            "sales": "actual"
        }
    )

    fold_output["lower_bound"] = lower_bound_fold
    fold_output["upper_bound"] = upper_bound_fold

    # Calculate the forecast horizon from day 1 to day 28
    fold_output["horizon"] = (
        pd.to_datetime(fold_output["forecast_date"]) - 
        fold_definitions[fold_number]["validation_start"]
    ).dt.days + 1

    # Save the predictions
    output_path = (
        PREDICTION_DIR /
        f"quantile_advanced_ca1_fold{fold_number}.parquet"
    )

    fold_output.to_parquet(output_path, index=False)

    # Verify the saved prediction data
    print("Prediction rows:", len(fold_output))
    print("Unique products:", fold_output["item_id"].nunique())
    print("Minimum horizon:", fold_output["horizon"].min())
    print("Maximum horizon:", fold_output["horizon"].max())
    print("Null predictions:",
          fold_output[["lower_bound", "upper_bound"]].isna().sum().sum())
    print("Crossed intervals:",
          (fold_output["lower_bound"] > fold_output["upper_bound"]).sum())
    print("Saved:", output_path)

print("\nFOLD 2 AND FOLD 3 QUANTILE PREDICTIONS COMPLETED!")


GENERATING QUANTILE PREDICTIONS: FOLD 2
--------------------------------
Prediction rows: 85372
Unique products: 3049
Minimum horizon: 1
Maximum horizon: 28
Null predictions: 0
Crossed intervals: 0
Saved: D:\Retail-Demand\predictions\quantile_advanced_ca1_fold2.parquet

GENERATING QUANTILE PREDICTIONS: FOLD 3
--------------------------------
Prediction rows: 85372
Unique products: 3049
Minimum horizon: 1
Maximum horizon: 28
Null predictions: 0
Crossed intervals: 0
Saved: D:\Retail-Demand\predictions\quantile_advanced_ca1_fold3.parquet

FOLD 2 AND FOLD 3 QUANTILE PREDICTIONS COMPLETED!


In [17]:
# CELL 16: Calculate conformal conformity scores using Fold 2

# Load Fold 2 quantile predictions
fold2_path = PREDICTION_DIR / "quantile_advanced_ca1_fold2.parquet"

fold2_calibration = pd.read_parquet(fold2_path)

# Verify required columns
required_columns = [
    "actual",
    "lower_bound",
    "upper_bound",
    "horizon"
]

assert all(
    column in fold2_calibration.columns
    for column in required_columns
), "Required columns are missing!"

# Calculate the conformity score for each observation
fold2_calibration["conformity_score"] = np.maximum.reduce([
    fold2_calibration["lower_bound"] - fold2_calibration["actual"],
    fold2_calibration["actual"] - fold2_calibration["upper_bound"],
    np.zeros(len(fold2_calibration))
])

# Calculate how many actual values were already covered
fold2_covered = (
    (fold2_calibration["actual"] >= fold2_calibration["lower_bound"]) &
    (fold2_calibration["actual"] <= fold2_calibration["upper_bound"])
)

# Calculate calibration statistics
score_values = fold2_calibration["conformity_score"].to_numpy()

print("FOLD 2 CONFORMAL CALIBRATION ANALYSIS")
print("--------------------------------")
print("Total calibration observations:", len(fold2_calibration))
print(f"Already covered: {fold2_covered.sum()}")
print(f"Outside intervals: {(~fold2_covered).sum()}")
print(f"Raw coverage: {fold2_covered.mean() * 100:.4f}%")

print("\nCONFORMITY SCORE SUMMARY")
print("--------------------------------")
print(f"Mean score: {np.mean(score_values):.6f}")
print(f"Median score: {np.median(score_values):.6f}")
print(f"90th percentile: {np.quantile(score_values, 0.90):.6f}")
print(f"95th percentile: {np.quantile(score_values, 0.95):.6f}")
print(f"Maximum score: {np.max(score_values):.6f}")
print(f"Zero-score percentage: {(score_values == 0).mean() * 100:.4f}%")

# Save calibration scores for later verification
calibration_score_path = (
    RESULTS_DIR / "conformal_scores_ca1_fold2.csv"
)

fold2_calibration[
    ["item_id", "forecast_date", "horizon", "conformity_score"]
].to_csv(calibration_score_path, index=False)

print("\nCALIBRATION SCORES SAVED:", calibration_score_path)
print("\nFOLD 2 CONFORMAL SCORE CALCULATION COMPLETED!")

FOLD 2 CONFORMAL CALIBRATION ANALYSIS
--------------------------------
Total calibration observations: 85372
Already covered: 80570
Outside intervals: 4802
Raw coverage: 94.3752%

CONFORMITY SCORE SUMMARY
--------------------------------
Mean score: 0.074925
Median score: 0.000000
90th percentile: 0.000000
95th percentile: 0.052247
Maximum score: 46.136710
Zero-score percentage: 94.3752%

CALIBRATION SCORES SAVED: D:\Retail-Demand\results\conformal_scores_ca1_fold2.csv

FOLD 2 CONFORMAL SCORE CALCULATION COMPLETED!


In [18]:
# CELL 17: Calculate the finite-sample conformal adjustment and apply it to Fold 3

# Set the desired nominal interval coverage
TARGET_COVERAGE = 0.90

# Extract the Fold 2 conformity scores
calibration_scores = fold2_calibration[
    "conformity_score"
].to_numpy()

# Calculate the finite-sample conformal rank
n_calibration = len(calibration_scores)

conformal_rank = int(
    np.ceil((n_calibration + 1) * TARGET_COVERAGE)
)

# Sort the calibration scores
sorted_scores = np.sort(calibration_scores)

# Calculate the conformal adjustment using the finite-sample rank
if conformal_rank > n_calibration:
    conformal_adjustment = np.inf
else:
    conformal_adjustment = sorted_scores[conformal_rank - 1]

print("CONFORMAL ADJUSTMENT")
print("--------------------------------")
print("Calibration observations:", n_calibration)
print("Target coverage:", TARGET_COVERAGE * 100, "%")
print("Finite-sample rank:", conformal_rank)
print(f"Conformal adjustment: {conformal_adjustment:.6f}")

# Load Fold 3 predictions
fold3_path = PREDICTION_DIR / "quantile_advanced_ca1_fold3.parquet"

fold3_evaluation = pd.read_parquet(fold3_path)

# Expand Fold 3 intervals using the calibration adjustment
fold3_evaluation["calibrated_lower"] = np.maximum(
    fold3_evaluation["lower_bound"] - conformal_adjustment,
    0
)

fold3_evaluation["calibrated_upper"] = (
    fold3_evaluation["upper_bound"] + conformal_adjustment
)

# Calculate raw Fold 3 coverage
raw_coverage = (
    (fold3_evaluation["actual"] >= fold3_evaluation["lower_bound"]) &
    (fold3_evaluation["actual"] <= fold3_evaluation["upper_bound"])
).mean() * 100

# Calculate calibrated Fold 3 coverage
calibrated_coverage = (
    (fold3_evaluation["actual"] >= fold3_evaluation["calibrated_lower"]) &
    (fold3_evaluation["actual"] <= fold3_evaluation["calibrated_upper"])
).mean() * 100

# Calculate average interval widths
raw_width = (
    fold3_evaluation["upper_bound"] -
    fold3_evaluation["lower_bound"]
).mean()

calibrated_width = (
    fold3_evaluation["calibrated_upper"] -
    fold3_evaluation["calibrated_lower"]
).mean()

# Display the evaluation results
print("\nFOLD 3 CONFORMAL EVALUATION")
print("--------------------------------")
print(f"Raw coverage: {raw_coverage:.4f}%")
print(f"Calibrated coverage: {calibrated_coverage:.4f}%")
print(f"Raw average width: {raw_width:.4f}")
print(f"Calibrated average width: {calibrated_width:.4f}")

# Save calibrated Fold 3 predictions
calibrated_path = (
    PREDICTION_DIR /
    "conformal_advanced_ca1_fold3.parquet"
)

fold3_evaluation.to_parquet(calibrated_path, index=False)

print("\nCALIBRATED PREDICTIONS SAVED:", calibrated_path)
print("Prediction rows:", len(fold3_evaluation))
print("\nCONFORMAL CALIBRATION COMPLETED!")

CONFORMAL ADJUSTMENT
--------------------------------
Calibration observations: 85372
Target coverage: 90.0 %
Finite-sample rank: 76836
Conformal adjustment: 0.000000

FOLD 3 CONFORMAL EVALUATION
--------------------------------
Raw coverage: 94.4959%
Calibrated coverage: 94.4959%
Raw average width: 4.3832
Calibrated average width: 4.3832

CALIBRATED PREDICTIONS SAVED: D:\Retail-Demand\predictions\conformal_advanced_ca1_fold3.parquet
Prediction rows: 85372

CONFORMAL CALIBRATION COMPLETED!


In [19]:
# CELL 18: Analyze conformal prediction coverage and interval width by forecast horizon

# Group Fold 3 predictions by forecast horizon
horizon_results = []

for horizon_day, group in fold3_evaluation.groupby("horizon"):

    # Extract actual sales and prediction intervals
    actual = group["actual"].to_numpy()

    raw_lower = group["lower_bound"].to_numpy()
    raw_upper = group["upper_bound"].to_numpy()

    calibrated_lower = group["calibrated_lower"].to_numpy()
    calibrated_upper = group["calibrated_upper"].to_numpy()

    # Calculate raw interval coverage
    raw_covered = (
        (actual >= raw_lower) &
        (actual <= raw_upper)
    )

    # Calculate calibrated interval coverage
    calibrated_covered = (
        (actual >= calibrated_lower) &
        (actual <= calibrated_upper)
    )

    # Calculate average interval widths
    raw_width = np.mean(raw_upper - raw_lower)

    calibrated_width = np.mean(
        calibrated_upper - calibrated_lower
    )

    # Store the results for this forecast horizon
    horizon_results.append({
        "horizon": int(horizon_day),
        "observations": len(group),
        "raw_coverage": raw_covered.mean() * 100,
        "calibrated_coverage": calibrated_covered.mean() * 100,
        "raw_average_width": raw_width,
        "calibrated_average_width": calibrated_width
    })

# Convert the results into a DataFrame
horizon_df = pd.DataFrame(horizon_results)

# Sort results from Day 1 to Day 28
horizon_df = horizon_df.sort_values("horizon").reset_index(drop=True)

# Display the horizon-wise results
print("HORIZON-WISE CONFORMAL COVERAGE ANALYSIS")
print("--------------------------------")
print(horizon_df.round(4).to_string(index=False))

# Display summary statistics
print("\nHORIZON ANALYSIS SUMMARY")
print("--------------------------------")
print("Number of forecast horizons:", len(horizon_df))
print("Minimum horizon:", horizon_df["horizon"].min())
print("Maximum horizon:", horizon_df["horizon"].max())

print(
    "Minimum calibrated coverage:",
    round(horizon_df["calibrated_coverage"].min(), 4), "%"
)

print(
    "Maximum calibrated coverage:",
    round(horizon_df["calibrated_coverage"].max(), 4), "%"
)

print(
    "Average calibrated coverage:",
    round(horizon_df["calibrated_coverage"].mean(), 4), "%"
)

# Save horizon-wise analysis
horizon_results_path = (
    RESULTS_DIR / "horizon_wise_conformal_coverage_ca1_fold3.csv"
)

horizon_df.to_csv(horizon_results_path, index=False)

print("\nHORIZON RESULTS SAVED:", horizon_results_path)
print("\nHORIZON-WISE ANALYSIS COMPLETED!")

HORIZON-WISE CONFORMAL COVERAGE ANALYSIS
--------------------------------
 horizon  observations  raw_coverage  calibrated_coverage  raw_average_width  calibrated_average_width
       1          3049       94.1620              94.1620             4.0889                    4.0889
       2          3049       94.3260              94.3260             3.8161                    3.8161
       3          3049       95.4739              95.4739             3.8155                    3.8155
       4          3049       93.8668              93.8668             3.8142                    3.8142
       5          3049       94.2276              94.2276             4.4824                    4.4824
       6          3049       94.8836              94.8836             5.1906                    5.1906
       7          3049       93.9324              93.9324             5.2410                    5.2410
       8          3049       94.3260              94.3260             4.3674                    4.3674

In [20]:
# CELL 19: Generate advanced LightGBM L2 and Tweedie predictions for Folds 2 and 3

# Store the trained models for convenient iteration
advanced_models = {
    "l2": l2_model,
    "tweedie": tweedie_model
}

# Generate predictions for Fold 2 and Fold 3
for fold_number in [2, 3]:

    print(f"\nGENERATING POINT PREDICTIONS: FOLD {fold_number}")
    print("--------------------------------")

    # Prepare the validation data using the same features as training
    fold_data, X_fold = prepare_quantile_fold(fold_number)

    # Generate predictions from both trained models
    for model_name, model in advanced_models.items():

        predictions = model.predict(
            X_fold,
            num_iteration=model.best_iteration_
        )

        # Demand predictions cannot be negative
        predictions = np.maximum(predictions, 0)

        # Prepare the output DataFrame
        output = fold_data[
            ["date", "sales", "item_id"]
        ].copy()

        output = output.rename(
            columns={
                "date": "forecast_date",
                "sales": "actual"
            }
        )

        output["prediction"] = predictions

        # Calculate the forecast horizon from Day 1 to Day 28
        output["horizon"] = (
            pd.to_datetime(output["forecast_date"]) -
            fold_definitions[fold_number]["validation_start"]
        ).dt.days + 1

        # Save the model predictions
        prediction_path = (
            PREDICTION_DIR /
            f"lightgbm_{model_name}_advanced_ca1_fold{fold_number}.parquet"
        )

        output.to_parquet(prediction_path, index=False)

        # Verify the generated predictions
        print(f"\nModel: {model_name.upper()}")
        print("Prediction rows:", len(output))
        print("Null predictions:", output["prediction"].isna().sum())
        print("Negative predictions:", (output["prediction"] < 0).sum())
        print("Minimum prediction:", round(output["prediction"].min(), 4))
        print("Maximum prediction:", round(output["prediction"].max(), 4))
        print("Saved:", prediction_path)

print("\nFOLD 2 AND FOLD 3 POINT PREDICTIONS COMPLETED!")


GENERATING POINT PREDICTIONS: FOLD 2
--------------------------------

Model: L2
Prediction rows: 85372
Null predictions: 0
Negative predictions: 0
Minimum prediction: 0.0784
Maximum prediction: 64.2539
Saved: D:\Retail-Demand\predictions\lightgbm_l2_advanced_ca1_fold2.parquet

Model: TWEEDIE
Prediction rows: 85372
Null predictions: 0
Negative predictions: 0
Minimum prediction: 0.0278
Maximum prediction: 85.6892
Saved: D:\Retail-Demand\predictions\lightgbm_tweedie_advanced_ca1_fold2.parquet

GENERATING POINT PREDICTIONS: FOLD 3
--------------------------------

Model: L2
Prediction rows: 85372
Null predictions: 0
Negative predictions: 0
Minimum prediction: 0.1651
Maximum prediction: 82.5625
Saved: D:\Retail-Demand\predictions\lightgbm_l2_advanced_ca1_fold3.parquet

Model: TWEEDIE
Prediction rows: 85372
Null predictions: 0
Negative predictions: 0
Minimum prediction: 0.0285
Maximum prediction: 96.093
Saved: D:\Retail-Demand\predictions\lightgbm_tweedie_advanced_ca1_fold3.parquet

FOLD 2

In [21]:
# CELL 20: Evaluate advanced L2 and Tweedie models across all three folds

# Import evaluation metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Store evaluation results
evaluation_results = []

# Evaluate both models across all three folds
for model_name in ["l2", "tweedie"]:

    for fold_number in [1, 2, 3]:

        # Load the corresponding prediction file
        prediction_path = (
            PREDICTION_DIR /
            f"lightgbm_{model_name}_advanced_ca1_fold{fold_number}.parquet"
        )

        prediction_data = pd.read_parquet(prediction_path)

        # Extract actual and predicted values
        actual = prediction_data["actual"].to_numpy()
        predicted = prediction_data["prediction"].to_numpy()

        # Calculate evaluation metrics
        mae = mean_absolute_error(actual, predicted)

        rmse = np.sqrt(
            mean_squared_error(actual, predicted)
        )

        wape = (
            np.sum(np.abs(actual - predicted)) /
            np.sum(np.abs(actual))
        ) * 100

        bias = (
            np.sum(predicted - actual) /
            np.sum(actual)
        ) * 100

        # Store the results
        evaluation_results.append({
            "model": model_name.upper(),
            "fold": fold_number,
            "MAE": mae,
            "RMSE": rmse,
            "WAPE": wape,
            "Bias": bias
        })

# Create evaluation DataFrame
evaluation_df = pd.DataFrame(evaluation_results)

# Display fold-wise comparison
print("ADVANCED MODEL PERFORMANCE BY FOLD")
print("--------------------------------")
print(evaluation_df.round(4).to_string(index=False))

# Calculate mean metrics across the three folds
average_results = (
    evaluation_df
    .groupby("model")[["MAE", "RMSE", "WAPE", "Bias"]]
    .mean()
    .reset_index()
)

print("\nAVERAGE MODEL PERFORMANCE")
print("--------------------------------")
print(average_results.round(4).to_string(index=False))

# Save detailed evaluation results
evaluation_path = (
    RESULTS_DIR / "advanced_model_evaluation_ca1.csv"
)

evaluation_df.to_csv(evaluation_path, index=False)

# Save average model comparison
average_path = (
    RESULTS_DIR / "advanced_model_average_comparison_ca1.csv"
)

average_results.to_csv(average_path, index=False)

print("\nDETAILED RESULTS SAVED:", evaluation_path)
print("AVERAGE COMPARISON SAVED:", average_path)
print("\nADVANCED MODEL EVALUATION COMPLETED!")

ADVANCED MODEL PERFORMANCE BY FOLD
--------------------------------
  model  fold    MAE   RMSE    WAPE    Bias
     L2     1 1.1100 2.2052 77.1289 -2.6981
     L2     2 1.1289 2.2159 76.9496 -2.2977
     L2     3 1.1290 2.1651 75.3286 -1.8998
TWEEDIE     1 1.0999 2.1974 76.4282 -3.1261
TWEEDIE     2 1.1127 2.1867 75.8427 -2.2472
TWEEDIE     3 1.1195 2.1667 74.6933 -0.9189

AVERAGE MODEL PERFORMANCE
--------------------------------
  model    MAE   RMSE    WAPE    Bias
     L2 1.1226 2.1954 76.4690 -2.2985
TWEEDIE 1.1107 2.1836 75.6548 -2.0974

DETAILED RESULTS SAVED: D:\Retail-Demand\results\advanced_model_evaluation_ca1.csv
AVERAGE COMPARISON SAVED: D:\Retail-Demand\results\advanced_model_average_comparison_ca1.csv

ADVANCED MODEL EVALUATION COMPLETED!


In [23]:
# CELL 21 (CORRECTED): Compare all four forecasting models for CA_1

# Store fresh comparison results
all_model_results = []

# Define the forecasting models
model_names = [
    "seasonal_naive",
    "croston_sba",
    "lightgbm_l2_advanced",
    "lightgbm_tweedie_advanced"
]

# Evaluate all models across three folds
for fold_number in [1, 2, 3]:

    print(f"\nEVALUATING FOLD {fold_number}")
    print("--------------------------------")

    for model_name in model_names:

        # Build the correct prediction file path
        if model_name in ["seasonal_naive", "croston_sba"]:

            prediction_path = (
                PREDICTION_DIR /
                f"{model_name}_fold_{fold_number}.parquet"
            )

        else:

            prediction_path = (
                PREDICTION_DIR /
                f"{model_name}_ca1_fold{fold_number}.parquet"
            )

        # Check whether the prediction file exists
        if not prediction_path.exists():
            raise FileNotFoundError(
                f"Prediction file not found: {prediction_path}"
            )

        # Load the prediction data
        prediction_data = pd.read_parquet(prediction_path)

        # Filter baseline predictions to CA_1
        if "store_id" in prediction_data.columns:
            prediction_data = prediction_data[
                prediction_data["store_id"] == "CA_1"
            ].copy()

        # Identify actual sales column
        if "actual" in prediction_data.columns:
            actual = prediction_data["actual"].to_numpy()
        elif "sales" in prediction_data.columns:
            actual = prediction_data["sales"].to_numpy()
        else:
            raise ValueError("Actual sales column not found!")

        # Identify prediction column
        if "prediction" in prediction_data.columns:
            predicted = prediction_data["prediction"].to_numpy()
        elif "forecast" in prediction_data.columns:
            predicted = prediction_data["forecast"].to_numpy()
        else:
            raise ValueError("Prediction column not found!")

        # Verify row counts
        assert len(actual) == len(predicted)
        assert len(actual) == 85372, (
            f"Unexpected row count for {model_name}, Fold {fold_number}"
        )

        # Calculate evaluation metrics
        mae = mean_absolute_error(actual, predicted)

        rmse = np.sqrt(
            mean_squared_error(actual, predicted)
        )

        wape = (
            np.sum(np.abs(actual - predicted)) /
            np.sum(np.abs(actual))
        ) * 100

        bias = (
            np.sum(predicted - actual) /
            np.sum(actual)
        ) * 100

        # Store evaluation results
        all_model_results.append({
            "model": model_name,
            "fold": fold_number,
            "MAE": mae,
            "RMSE": rmse,
            "WAPE": wape,
            "Bias": bias
        })

        print(
            f"{model_name}: "
            f"WAPE={wape:.4f}%, "
            f"MAE={mae:.4f}"
        )

# Create detailed comparison table
all_model_df = pd.DataFrame(all_model_results)

# Calculate average performance across three folds
all_model_average = (
    all_model_df
    .groupby("model")[["MAE", "RMSE", "WAPE", "Bias"]]
    .mean()
    .reset_index()
    .sort_values("WAPE")
)

# Display detailed results
print("\nALL MODELS - FOLD-WISE PERFORMANCE")
print("--------------------------------")
print(all_model_df.round(4).to_string(index=False))

# Display average results
print("\nAVERAGE MODEL COMPARISON")
print("--------------------------------")
print(all_model_average.round(4).to_string(index=False))

# Save detailed comparison
all_model_path = (
    RESULTS_DIR / "all_models_fold_comparison_ca1.csv"
)

all_model_df.to_csv(all_model_path, index=False)

# Save average comparison
all_model_average_path = (
    RESULTS_DIR / "all_models_average_comparison_ca1.csv"
)

all_model_average.to_csv(
    all_model_average_path,
    index=False
)

print("\nDETAILED COMPARISON SAVED:", all_model_path)
print("AVERAGE COMPARISON SAVED:", all_model_average_path)

print("\nALL FOUR MODEL COMPARISONS COMPLETED!")


EVALUATING FOLD 1
--------------------------------
seasonal_naive: WAPE=87.3350%, MAE=1.2568
croston_sba: WAPE=81.0362%, MAE=1.1662
lightgbm_l2_advanced: WAPE=77.1289%, MAE=1.1100
lightgbm_tweedie_advanced: WAPE=76.4282%, MAE=1.0999

EVALUATING FOLD 2
--------------------------------
seasonal_naive: WAPE=86.9265%, MAE=1.2753
croston_sba: WAPE=79.7198%, MAE=1.1695
lightgbm_l2_advanced: WAPE=76.9496%, MAE=1.1289
lightgbm_tweedie_advanced: WAPE=75.8427%, MAE=1.1127

EVALUATING FOLD 3
--------------------------------
seasonal_naive: WAPE=88.7554%, MAE=1.3303
croston_sba: WAPE=79.6000%, MAE=1.1930
lightgbm_l2_advanced: WAPE=75.3286%, MAE=1.1290
lightgbm_tweedie_advanced: WAPE=74.6933%, MAE=1.1195

ALL MODELS - FOLD-WISE PERFORMANCE
--------------------------------
                    model  fold    MAE   RMSE    WAPE    Bias
           seasonal_naive     1 1.2568 2.7483 87.3350 -6.9104
              croston_sba     1 1.1662 2.3687 81.0362 -0.7856
     lightgbm_l2_advanced     1 1.1100 2.20

In [24]:
# CELL 22: Verify all important Thursday implementation outputs

# Define the important output files to verify
required_files = [
    # Advanced feature dataset
    PROCESSED_DATA_DIR / "ca1_advanced_features.parquet",

    # Trained models
    MODEL_DIR / "lightgbm_l2_advanced_ca1_fold1.txt",
    MODEL_DIR / "lightgbm_tweedie_advanced_ca1_fold1.txt",
    MODEL_DIR / "lightgbm_quantile_0.05_advanced_ca1_fold1.txt",
    MODEL_DIR / "lightgbm_quantile_0.95_advanced_ca1_fold1.txt",

    # L2 predictions
    PREDICTION_DIR / "lightgbm_l2_advanced_ca1_fold1.parquet",
    PREDICTION_DIR / "lightgbm_l2_advanced_ca1_fold2.parquet",
    PREDICTION_DIR / "lightgbm_l2_advanced_ca1_fold3.parquet",

    # Tweedie predictions
    PREDICTION_DIR / "lightgbm_tweedie_advanced_ca1_fold1.parquet",
    PREDICTION_DIR / "lightgbm_tweedie_advanced_ca1_fold2.parquet",
    PREDICTION_DIR / "lightgbm_tweedie_advanced_ca1_fold3.parquet",

    # Quantile predictions
    PREDICTION_DIR / "quantile_advanced_ca1_fold1.parquet",
    PREDICTION_DIR / "quantile_advanced_ca1_fold2.parquet",
    PREDICTION_DIR / "quantile_advanced_ca1_fold3.parquet",

    # Conformal predictions and calibration scores
    PREDICTION_DIR / "conformal_advanced_ca1_fold3.parquet",
    RESULTS_DIR / "conformal_scores_ca1_fold2.csv",

    # Horizon-wise analysis
    RESULTS_DIR / "horizon_wise_conformal_coverage_ca1_fold3.csv",

    # Model evaluation results
    RESULTS_DIR / "advanced_model_evaluation_ca1.csv",
    RESULTS_DIR / "advanced_model_average_comparison_ca1.csv",
    RESULTS_DIR / "all_models_fold_comparison_ca1.csv",
    RESULTS_DIR / "all_models_average_comparison_ca1.csv"
]

# Check every required file
verified_count = 0
missing_files = []

print("THURSDAY OUTPUT VERIFICATION")
print("--------------------------------")

for file_path in required_files:

    if file_path.exists():

        file_size = file_path.stat().st_size / (1024 * 1024)

        print(
            f"[OK] {file_path.name} "
            f"| {file_size:.2f} MB"
        )

        verified_count += 1

    else:

        print(f"[MISSING] {file_path.name}")
        missing_files.append(str(file_path))

# Display final verification summary
print("\nVERIFICATION SUMMARY")
print("--------------------------------")
print("Total required files:", len(required_files))
print("Verified files:", verified_count)
print("Missing files:", len(missing_files))

if len(missing_files) == 0:
    print("\nALL REQUIRED THURSDAY FILES VERIFIED!")
else:
    print("\nSOME FILES NEED ATTENTION.")
    print("Check the missing file paths above.")

THURSDAY OUTPUT VERIFICATION
--------------------------------
[OK] ca1_advanced_features.parquet | 72.41 MB
[OK] lightgbm_l2_advanced_ca1_fold1.txt | 0.52 MB
[OK] lightgbm_tweedie_advanced_ca1_fold1.txt | 1.00 MB
[OK] lightgbm_quantile_0.05_advanced_ca1_fold1.txt | 1.48 MB
[OK] lightgbm_quantile_0.95_advanced_ca1_fold1.txt | 1.83 MB
[OK] lightgbm_l2_advanced_ca1_fold1.parquet | 0.66 MB
[OK] lightgbm_l2_advanced_ca1_fold2.parquet | 0.67 MB
[OK] lightgbm_l2_advanced_ca1_fold3.parquet | 0.67 MB
[OK] lightgbm_tweedie_advanced_ca1_fold1.parquet | 0.84 MB
[OK] lightgbm_tweedie_advanced_ca1_fold2.parquet | 0.84 MB
[OK] lightgbm_tweedie_advanced_ca1_fold3.parquet | 0.85 MB
[OK] quantile_advanced_ca1_fold1.parquet | 0.86 MB
[OK] quantile_advanced_ca1_fold2.parquet | 0.86 MB
[OK] quantile_advanced_ca1_fold3.parquet | 0.85 MB
[OK] conformal_advanced_ca1_fold3.parquet | 1.63 MB
[OK] conformal_scores_ca1_fold2.csv | 2.71 MB
[OK] horizon_wise_conformal_coverage_ca1_fold3.csv | 0.00 MB
[OK] advanced_